# BoxCommentator: vLLM benchmark on Kaggle

Runs the experiment plan from `serving/run_matrix.py` on this notebook's GPU and the hosted baselines
(Groq, Nebius), then zips `bench/results/` for download.

**Before you start**
1. Notebook settings: Accelerator = **GPU T4** (T4 x2 only if you want the optional fp16 7B), Internet = **On**.
2. Add-ons > Secrets: `GROQ_API_KEY`, `NEBIUS_API_KEY`, `NEBIUS_MODEL`
   (optional: `NEBIUS_BASE_URL`, default is in `config/backends.yaml`).
3. Set `REPO_URL` below.

**Cost / quota safety**: every hosted run below has `--max-requests` or `--max-usd` and `--rpm`.
Groq's free tier is about 30 requests/min and about 100K tokens/day for the 70B model, so Groq gets a small sample.
Kaggle GPU time is limited per week; run the sections one at a time.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <- set this
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd repo

In [ ]:
# VERIFY: vLLM must match this image's CUDA / torch. If the install changes torch, restart the kernel
# (Run > Restart) and continue from the next cell.
!pip install -q vllm pyyaml openai matplotlib

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["GROQ_API_KEY", "NEBIUS_API_KEY", "NEBIUS_MODEL", "NEBIUS_BASE_URL"]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print("loaded", name)
    except Exception:
        print("not set:", name)
os.environ.setdefault("NEBIUS_BASE_URL", "https://api.tokenfactory.nebius.com/v1/")

## 1. Show the plan (runs nothing)

In [ ]:
!python -m serving.run_matrix --plan core --fights 10 --dry-plan
!python -m bench.bench --list

## 2. Smoke test: one server, 2 short fights (a few minutes)

In [ ]:
!python -m serving.run_matrix --plan smoke --fights 2

## 3. Core experiments (self-hosted)
Quantization, prefix caching, batching and context capping. Servers are started and stopped one at a time;
a failing variant is skipped and reported at the end. Expect an hour or more.

In [ ]:
!python -m serving.run_matrix --plan core --fights 10

## 4. Hosted baselines (capped)

In [ ]:
# Groq free tier: ~150 requests/day on the 70B model => about 3 fights. Pace below the 30 req/min limit.
!python -m bench.bench --backend groq-70b --fights 3 --tag baseline --max-requests 150 --rpm 25

# Nebius: capped by estimated spend. Fill the prices in config/backends.yaml first, or --max-usd cannot track cost.
!python -m bench.bench --backend nebius --fights 10 --tag baseline --max-usd 0.20 --rpm 120

## 5. Routing: knockdowns to the fastest backend, analysis to the strongest
Starts the 3B server, then compares routed vs single-model on the same fights.

In [ ]:
import os
from serving.run_matrix import Server
from bench.bench import RunConfig, run_bench, print_summary

with Server("3b") as srv:
    os.environ["VLLM_BASE_URL"] = srv.base_url
    cfg = RunConfig(routing=True, track_a="groq-70b", track_b="vllm-3b", fallback=["groq-70b"],
                    tag="routing", fights=3, max_requests=150, rpm=25)
    print_summary(run_bench(cfg)["summary"])

## 6. Charts and download

In [ ]:
!python -m bench.charts
import shutil
shutil.make_archive("/kaggle/working/bench_results", "zip", "bench/results")
print("download /kaggle/working/bench_results.zip from the Output tab")